In [2]:
import torch
import numpy as np
import torch.nn as nn

# Hamiltonian Mechanics Recap

In Hamiltonian mechanics:

$$
x = \begin{bmatrix} q \ p \end{bmatrix}, \quad q \in \mathbb{R}^{n}, p \in \mathbb{R}^{n}
$$

* (q) is the generalized coordinate (position)
* (p) is the generalized momentum
* The **Hamiltonian** (H(q,p)) represents the total energy:

$$
H(q, p) = T(p, q) + V(q)
$$

where:

* $V(q)$ is the **potential energy**.
* $T(p,q) = \frac{1}{2} p^\top M^{-1}(q) p$ is the **kinetic energy**, and $M(q)$ is the **inertia/mass matrix** (SPD).

The **Hamiltonian equations of motion** are:

$$
\dot{q} = \frac{\partial H}{\partial p}, \quad \dot{p} = -\frac{\partial H}{\partial q}
$$

# MLP: Direct Dynamics Learning

This network **directly learns the time derivative** of the system state:

$$
x = \begin{bmatrix} q \ p \end{bmatrix} \in \mathbb{R}^{2n}, \quad \dot{x} = f_\theta(x)
$$

* $f_\theta$ is the neural network (`MLP`)
* Input: full state $x$
* Output: derivative $\dot{x} \approx f_\text{true}(x)$


## Network Structure

$$
\text{MLP}: \mathbb{R}^{2n} \to \mathbb{R}^{2n}, \quad \text{with 5 hidden layers, Tanh activations}
$$

* Fully connected layers
* Output predicts $\dot{q}$ and $\dot{p}$ directly


## Key Points

* **No physical bias enforced**
* **No guaranteed energy or symplecticity preservation**



In [3]:
class MLP(nn.Module):
    """Baseline MLP that directly learns dynamics"""
    def __init__(self, input_dim=2, hidden_dim=64):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim)  # Output is derivative
        )

    def forward(self, x):
        return self.net(x)

# Hamiltonian Neural Network (HNN)

This network **learns the Hamiltonian** $H(q,p)$ of a system and computes dynamics via Hamilton’s equations:

$$
x = \begin{bmatrix} q \ p \end{bmatrix} \in \mathbb{R}^{2n}, \quad H_\theta(x) \in \mathbb{R}
$$

## 1. Hamiltonian Network

$$
H_\theta(x) = \text{MLP}(x) : \mathbb{R}^{2n} \to \mathbb{R}
$$

* Fully connected MLP with `Tanh` activations
* Outputs a **scalar Hamiltonian**


## 2. Dynamics via Gradients

Hamilton’s equations:

$$
\dot{q} = \frac{\partial H}{\partial p}, \quad \dot{p} = -\frac{\partial H}{\partial q}
$$

Implemented with automatic differentiation:

```python
grad_H = torch.autograd.grad(H.sum(), x, create_graph=True)[0]
dq_dt = grad_H[:, n:]     # ∂H/∂p
dp_dt = -grad_H[:, :n]    # -∂H/∂q
```

* Returns $\dot{x} = [\dot{q}, \dot{p}]$


## 3. Key Points

* Does **not explicitly parametrize mass matrix**
* More physically informed than direct MLP.

In [4]:
class HNN(nn.Module):
    """Hamiltonian Neural Network"""
    def __init__(self, input_dim=2, hidden_dim=64):
        super(HNN, self).__init__()
        self.input_dim = input_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)  # Output is scalar Hamiltonian
        )

    def forward(self, x):
        return self.net(x)

    def get_dynamics(self, x):
        """Compute dynamics using automatic differentiation"""
        # Enable gradient computation for this specific operation
        with torch.enable_grad():
            x = x.detach().requires_grad_(True)
            H = self.forward(x)

            # Compute gradients
            grad_H = torch.autograd.grad(H.sum(), x, create_graph=True)[0]

            # Hamiltonian dynamics: dq/dt = ∂H/∂p, dp/dt = -∂H/∂q
            q_dim = self.input_dim // 2
            dq_dt = grad_H[:, q_dim:]   # ∂H/∂p
            dp_dt = -grad_H[:, :q_dim]  # -∂H/∂q

            return torch.cat([dq_dt, dp_dt], dim=1)

# DoubleHeadHNN: Separate Kinetic and Potential Networks

This network **learns the Hamiltonian** by **modeling kinetic and potential energy separately**:

$$
H(q,p) = T(q,p) + V(q)
$$

where:

* $V(q)$ is potential energy
* $T(q,p) = \frac{1}{2} p^\top M^{-1}(q) p$ is kinetic energy


## 1. Potential Energy Network

$$
V_\theta(q) : \mathbb{R}^n \to \mathbb{R}, \quad \text{MLP outputs scalar } V
$$


## 2. Inverse Mass Matrix Network

$$
M^{-1}_\theta(q) : \mathbb{R}^n \to \mathbb{R}^{n \times n}
$$

* Flattened output reshaped to (n \times n)
* Symmetrized: $M^{-1} \gets 0.5 (M^{-1} + (M^{-1})^\top)$
* Small diagonal added $(1e^{-3} I)$ for numerical stability and positive definiteness


## 3. Kinetic Energy

$$
T(q,p) = \frac{1}{2} p^\top M^{-1}(q) p
$$

## 4. Dynamics via Gradients

Hamilton’s equations:

$$
\dot{q} = \frac{\partial H}{\partial p}, \quad \dot{p} = -\frac{\partial H}{\partial q}
$$

* Computed with automatic differentiation
* Returns $\dot{x} = [\dot{q}, \dot{p}]$


## 5. Key Points
* **Separates T and V for interpretability**
* Directly predicts $M^{-1}(q)$ (symmetric + positive definite)
* Ensures **energy-conserving dynamics**
* More flexible than standard HNN, but requires care to maintain SPD property.


In [5]:
class DoubleHeadHNN(nn.Module):
    """DoubleHeadHNN: Separate networks for kinetic (T) and potential (V) energy"""
    def __init__(self, input_dim=2, hidden_dim=64):
        super(DoubleHeadHNN, self).__init__()
        self.q_dim = input_dim // 2
        self.p_dim = input_dim // 2

        # Network for potential energy V(q)
        self.v_net = nn.Sequential(
            nn.Linear(self.q_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        # Network for INVERSE inertia matrix M^{-1}(q)
        # Output size: q_dim x q_dim matrix (flattened)
        self.m_inv_net = nn.Sequential(
            nn.Linear(self.q_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, self.q_dim * self.q_dim)
        )

    def forward(self, x):
        """Compute Hamiltonian H = T + V"""
        q = x[:, :self.q_dim]
        p = x[:, self.q_dim:]

        # Potential energy
        V = self.v_net(q)

        # Kinetic energy: T = 0.5 * p^T * M^{-1}(q) * p
        M_inv_flat = self.m_inv_net(q)
        M_inv = M_inv_flat.view(-1, self.q_dim, self.q_dim)

        # Make M_inv symmetric
        M_inv = 0.5 * (M_inv + M_inv.transpose(-1, -2))

        # Add small diagonal term for numerical stability (positive definiteness)
        eye = torch.eye(self.q_dim, device=M_inv.device).unsqueeze(0)
        M_inv = M_inv + 1e-3 * eye

        # Compute M^{-1} * p
        p_expanded = p.unsqueeze(-1)
        M_inv_p = torch.bmm(M_inv, p_expanded).squeeze(-1)

        # Kinetic energy
        T = 0.5 * torch.sum(p * M_inv_p, dim=1, keepdim=True)

        return T + V

    def get_dynamics(self, x):
        """Compute dynamics using automatic differentiation"""
        # Enable gradient computation for this specific operation
        with torch.enable_grad():
            x = x.detach().requires_grad_(True)
            H = self.forward(x)

            # Compute gradients
            grad_H = torch.autograd.grad(H.sum(), x, create_graph=True)[0]

            # Split gradients for q and p dimensions
            q_dim = self.q_dim
            dH_dq = grad_H[:, :q_dim]
            dH_dp = grad_H[:, q_dim:]

            # Hamiltonian dynamics
            dq_dt = dH_dp
            dp_dt = -dH_dq

            return torch.cat([dq_dt, dp_dt], dim=1)

# CholeskyHNN: Hamiltonian Neural Network with SPD Mass Matrix

This network is a **Hamiltonian Neural Network (HNN)** that learns the dynamics of a physical system while ensuring that the **mass (inertia) matrix is symmetric positive definite (SPD)** using a **Cholesky decomposition**.


## 1. Potential Energy Network

The potential energy is modeled as a neural network:

$$
V_\theta(q) : \mathbb{R}^n \to \mathbb{R}
$$

```python
self.v_net = nn.Sequential(
    nn.Linear(self.q_dim, hidden_dim),
    nn.Tanh(),
    nn.Linear(hidden_dim, hidden_dim),
    nn.Tanh(),
    nn.Linear(hidden_dim, 1)
)
```

This network learns **arbitrary potential landscapes** from data.

## 2. Cholesky Parameterization of the Mass Matrix

The mass matrix $M(q)$ must be **symmetric positive definite (SPD)**.

To guarantee this, we use a **Cholesky decomposition**:

$$
M(q) = L(q) L(q)^\top
$$

where $L(q)$ is a **lower triangular matrix with positive diagonals**.

* Network outputs parameters of $L(q)$ (lower-triangular elements):

$$
l_\theta(q) \in \mathbb{R}^{n(n+1)/2} \quad \text{(all entries of lower triangular matrix)}
$$

* The diagonal is exponentiated to enforce positivity:

$$
L_{ii} = \exp(l_i) > 0
$$

* Then the kinetic energy is:

$$
T(q, p) = \frac{1}{2} p^\top M^{-1}(q) p = \frac{1}{2} p^\top (L L^\top)^{-1} p
$$


### 3. Computing $M^{-1} p$ Efficiently

Instead of explicitly inverting $M$, we solve:

1. $L y = p \implies y = L^{-1} p$
2. $L^\top z = y \implies z = (L^\top)^{-1} y = M^{-1} p$

Then:

$$
T = \frac{1}{2} p^\top z
$$

This is **numerically stable** and avoids direct matrix inversion.


## 4. Forward Pass

Given input $x = [q, p]$:

1. Split positions and momenta:

$$
q = x[:, :n], \quad p = x[:, n:]
$$

2. Compute potential energy: $V(q)$
3. Compute Cholesky factor $L(q)$
4. Solve for $z = M^{-1} p$
5. Compute kinetic energy $T = \frac{1}{2} p^\top z$
6. Return **Hamiltonian**:

$$
H(q,p) = T + V
$$


## 5. Dynamics via Automatic Differentiation

The network computes **Hamiltonian dynamics** using gradients:

$$
\dot{q} = \frac{\partial H}{\partial p}, \quad \dot{p} = -\frac{\partial H}{\partial q}
$$

```python
grad_H = torch.autograd.grad(H.sum(), x, create_graph=True)[0]
dq_dt = grad_H[:, :n]
dp_dt = -grad_H[:, n:]
```

* `dq_dt` is the velocity
* `dp_dt` is the momentum derivative
* Returns $[\dot{q}, \dot{p}]$



In [6]:
class CholeskyHNN(nn.Module):
    """HNN with Cholesky decomposition to ensure SPD inertia matrix"""
    def __init__(self, input_dim=2, hidden_dim=64):
        super(CholeskyHNN, self).__init__()
        self.q_dim = input_dim // 2
        self.p_dim = input_dim // 2

        # Network for potential energy V(q)
        self.v_net = nn.Sequential(
            nn.Linear(self.q_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        # Network for Cholesky factor L(q)
        # Output size: lower triangular entries of L
        self.l_net = nn.Sequential(
            nn.Linear(self.q_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, self.q_dim * (self.q_dim + 1) // 2)
        )

    def _construct_lower_triangular(self, l_params):
        """Construct lower triangular matrix from parameters"""
        batch_size = l_params.shape[0]
        L = torch.zeros(batch_size, self.q_dim, self.q_dim, device=l_params.device)

        # Fill lower triangular part
        idx = 0
        for i in range(self.q_dim):
            for j in range(i + 1):
                if i == j:
                    # Ensure positive diagonal elements
                    L[:, i, j] = torch.exp(l_params[:, idx])
                else:
                    L[:, i, j] = l_params[:, idx]
                idx += 1

        return L

    def forward(self, x):
        """Compute Hamiltonian H = T + V"""
        q = x[:, :self.q_dim]
        p = x[:, self.q_dim:]

        # Potential energy
        V = self.v_net(q)

        # Kinetic energy: T = 0.5 * p^T * M^{-1}(q) * p
        # where M = L * L^T (Cholesky decomposition)
        l_params = self.l_net(q)
        L = self._construct_lower_triangular(l_params)

        # Solve L * y = p for y
        p_expanded = p.unsqueeze(-1)
        y = torch.linalg.solve_triangular(L, p_expanded, upper=False).squeeze(-1)

        # Solve L^T * z = y for z = M^{-1} * p
        z = torch.linalg.solve_triangular(L.transpose(-1, -2), y.unsqueeze(-1), upper=True).squeeze(-1)

        # Kinetic energy
        T = 0.5 * torch.sum(p * z, dim=1, keepdim=True)

        return T + V

    def get_dynamics(self, x):
        """Compute dynamics using automatic differentiation"""
        # Enable gradient computation for this specific operation
        with torch.enable_grad():
            x = x.detach().requires_grad_(True)
            H = self.forward(x)

            # Compute gradients
            grad_H = torch.autograd.grad(H.sum(), x, create_graph=True)[0]

            # Split gradients for q and p dimensions
            q_dim = self.q_dim
            dH_dq = grad_H[:, :q_dim]
            dH_dp = grad_H[:, q_dim:]

            # Hamiltonian dynamics
            dq_dt = dH_dp
            dp_dt = -dH_dq

            return torch.cat([dq_dt, dp_dt], dim=1)